# Quantum walks

$\renewcommand{\ket}[1]{\left|#1\right\rangle}\renewcommand{\bra}[1]{\left\langle #1\right|}\renewcommand{\braket}[2]{\left\langle #1 \middle| #2 \right\rangle}\renewcommand{\ketbra}[2]{\left|#1\right\rangle\!\left\langle #2\right|}$

This notebook describes the Szegedy quantum walk operator used in the paper and summarizes the logical resources required by its circuit implementation. The notation and resource formulas follow the main text and Appendices E and F of the manuscript.

**Table of contents**

1. [Szegedy quantum walk operator](#szegedy-quantum-walk-operator)
2. [Summary of the resources](#summary-of-the-resources)
3. [Boltzmann coin](#boltzmann-coin)
4. [Proposal preparation unitaries](#proposal-preparation-unitaries)
5. [Reflection and accept-path unitary](#reflection-and-accept-path-unitary)
6. [Comparison with Lemieux et al.](#comparison-with-lemieux-et-al)


<a id="szegedy-quantum-walk-operator"></a>
## Szegedy quantum walk operator

Consider an ergodic reversible Markov chain on $n$ Ising spins, with transition matrix $P_\beta\in\mathbb{R}^{2^n\times2^n}$. For a symmetric proposal matrix $T$, the off-diagonal entries are

$$
(P_\beta)_{yx}
=
T_{yx}A_{yx}^{(\beta)},
\qquad
y\neq x,
$$

where

$$
A_{yx}^{(\beta)}
=
\min\left\{
1,
\exp\left[-\beta\left(H(y)-H(x)\right)\right]
\right\},
$$

and the diagonal entries are fixed by normalization.

The walk acts on $\mathcal H_a\otimes\mathcal H_b\otimes\mathcal H_c$. Registers $a$ and $b$ store the current and proposed configurations, while register $c$ is the Metropolis coin. The half-step Szegedy walk used in the paper is

$$
W_\beta
=
R_0V^\dagger B_\beta^\dagger F B_\beta V.
$$

The proposal-preparation unitary is

$$
V\ket{x}_a\ket{0^n}_b
=
\ket{x}_a
\sum_y\sqrt{T_{yx}}\ket{y}_b.
$$

The abstract one-qubit Boltzmann coin is

$$
B_\beta\ket{x}_a\ket{y}_b\ket{0}_c
=
\ket{x}_a\ket{y}_b
\left(
\sqrt{1-A_{yx}^{(\beta)}}\ket{0}_c
+
\sqrt{A_{yx}^{(\beta)}}\ket{1}_c
\right),
$$

where $\ket{1}_c$ denotes acceptance and $\ket{0}_c$ denotes rejection. The accept-path unitary swaps the two configuration registers only on the accepted branch:

$$
F\ket{x}_a\ket{y}_b\ket{0}_c
=
\ket{x}_a\ket{y}_b\ket{0}_c,
$$

$$
F\ket{x}_a\ket{y}_b\ket{1}_c
=
\ket{y}_a\ket{x}_b\ket{1}_c.
$$

The reflection about the joint zero state of the proposal and coin registers is

$$
R_0
=
2\left(
I_a\otimes\ket{0^n}\!\bra{0^n}_b\otimes\ket{0}\!\bra{0}_c
\right)-I.
$$

Define

$$
U_\beta
=
V^\dagger B_\beta^\dagger F B_\beta V.
$$

The zero block of both $U_\beta$ and $W_\beta$ is the discriminant matrix $X_\beta$:

$$
\left(
I_a\otimes\bra{0^n}_b\bra{0}_c
\right)
W_\beta
\left(
I_a\otimes\ket{0^n}_b\ket{0}_c
\right)
=
X_\beta.
$$

For a reversible chain,

$$
(X_\beta)_{xy}
=
\sqrt{(P_\beta)_{xy}(P_\beta)_{yx}},
$$

and $X_\beta$ is similar to $P_\beta$, so they have the same spectrum. The coherent Gibbs state

$$
\ket{\pi_\beta}
=
\sum_x\sqrt{\pi_\beta(x)}\ket{x}
$$

is the $+1$ eigenstate associated with the stationary eigenvalue. If $\lambda\in\operatorname{spec}(X_\beta)$, the corresponding walk eigenvalues are

$$
\exp\left[\pm i\arccos(\lambda)\right].
$$

For the Hamiltonian-simulation proposal, $V$ prepares phaseful amplitudes rather than the positive amplitudes $\sqrt{T_{yx}}$. Appendix E shows that the same discriminant block is obtained because the simulated transverse-field Hamiltonian is symmetric in the computational basis, so the phases cancel in the relevant overlaps.


<a id="summary-of-the-resources"></a>
## Summary of the resources

The resource model tracks the logical non-Clifford depth and the number of logical qubits. If $D_V$, $D_B$, $D_F$, and $D_R$ denote the depths of the proposal preparation, Boltzmann coin, accept-path unitary, and reflection, respectively, then one walk application has depth

$$
D_W
=
2D_V+2D_B+D_F+D_R.
$$

The asymptotic resource estimates in Table II of the paper are:

| Component | Logical non-Clifford depth | Logical qubits |
|---|---:|---:|
| Uniform proposal | $0$ | $2n$ |
| Local proposal, $k=1$ spin flip | $O(\log n)$ | $2n$ |
| Hamiltonian-simulation proposal, $r$ Trotter steps | $O(rn)$ | $2n$ |
| Boltzmann coin, phase arithmetic | $O(n^2)$ | $O(n)$ |
| Boltzmann coin, hybrid arithmetic | $O(\log n\log\varepsilon^{-1})$ | $O(n^2\log(n/\varepsilon))$ |
| Reflection $R_0$ | $O(\log(n+\log\varepsilon^{-1}))$ | $O(n+\log\varepsilon^{-1})$ |
| Accept-path unitary | $O(\log\log(n/\varepsilon))$ | $O(n+\log(n/\varepsilon))$ |

The explicit circuit formulas use

$$
M
=
n+\binom{n}{2}
=
\frac{n(n+1)}{2},
\qquad
\ell_x
=
\left\lceil\log_2x\right\rceil.
$$

All primitive gates are recursively expanded into the Clifford+$T$+$R_z$ gate set rather than being decomposed through generic transpilation. The exact primitive resources in Table III are:

| Gate | Logical qubits | Non-Clifford depth |
|---|---:|---:|
| $\mathrm{CCX}$ | $3$ | $3$ |
| $\mathrm{CR}_y(\theta)$ | $2$ | $2$ |
| $\mathrm{CCR}_y(\theta)$ | $3$ | $8$ |
| $G(\theta)$ | $2$ | $2$ |
| $\mathrm{C}G(\theta)$ | $3$ | $14$ |

The non-Clifford depth counts layers of $T$, $T^\dagger$, and arbitrary $R_z$ rotations after this recursive expansion.


<a id="boltzmann-coin"></a>
## Boltzmann coin

The Boltzmann coin applies the square-root Metropolis amplitude. The paper develops two implementations: fully phase-based arithmetic and hybrid fixed-point and phase arithmetic.

### Fully phase-based arithmetic

The fully phase-based construction encodes the energy difference in the spectrum of

$$
H_{\mathrm{ph}}
=
-H^{(a)}+H^{(b)},
$$

and applies the acceptance function through GQSP. It is the most space-efficient implementation, but its depth scales quadratically with $n$.

Let $d_{\mathrm{ph}}$ be the GQSP approximation parameter. Table V gives:

| Component | Logical qubits | Non-Clifford depth |
|---|---:|---:|
| $\mathrm{PREPARE}$ | $6n$ | $60n-56$ |
| $\mathrm{SELECT}$ | $8n$ | $0$ |
| controlled-$\mathrm{SELECT}$ | $10n$ | $6n+9$ |
| $R_0$ | $6n+2$ | $14\ell_{6n}-10$ |
| controlled-$R_0$ | $6n+3$ | $14\ell_{6n}-10$ |
| $Q_{\mathrm{ph}}$ | $8n+2$ | $120n+14\ell_{6n}-122$ |
| controlled-$Q_{\mathrm{ph}}$ | $10n+2$ | $126n+14\ell_{6n}-113$ |
| Fully phase arithmetic | $10n+2$ | $3d_{\mathrm{ph}}(126n+14\ell_{6n}-113)+3(2d_{\mathrm{ph}}+1)$ |

Writing

$$
\ell_\varepsilon
=
\left\lceil
\log_2(\varepsilon_{\mathrm{ops}}^{-1})
\right\rceil,
$$

the degree estimate used in the paper is

$$
d_{\mathrm{ph}}
=
2+
\left\lceil
\max\left\{
\sqrt{\frac{\beta B}{2}\ell_\varepsilon}+\ell_\varepsilon,
\frac{B}{\omega}\ell_\varepsilon
\right\}
\right\rceil,
$$

where $B=\|H_{\mathrm{ph}}\|\approx2n$ and $\omega$ is the constant-width interval excluded around the kink at $\Delta H=0$.

This implementation is detailed in `3.1_fully_phase_arithmetic.ipynb`.

### Hybrid fixed-point and phase arithmetic

The hybrid construction first computes the signed energy difference using fixed-point arithmetic. It then selects $(\Delta H)_+$, applies the cutoff-tail transformation, and implements the smooth square-root exponential using one-body qubitization and GQSP.

Let $w=b+1$ be the signed fixed-point word size and define

$$
s_{\mathrm{SK}}
=
\left\lceil
\log_{3/2}(2M)
\right\rceil.
$$

The implemented cutoff power is

$$
j_{\mathrm{cut}}
=
\max\left\{
0,
\min\left[
w-1,
\left\lfloor
\log_2\left(
\frac{\beta\Lambda}
{2\log(1/\varepsilon_{\mathrm{tail}})}
\right)
\right\rfloor
\right]
\right\}.
$$

Table VI gives the resources for the fixed-point part:

| Component | Logical qubits | Non-Clifford depth |
|---|---:|---:|
| Conditional-terms loader | $Mw+2M-n$ | $0$ |
| Three-to-two compressor | $5$ | $9$ |
| Wallace-tree adder | $6Mw-2w-2M+1$ | $18s_{\mathrm{SK}}+18w$ |
| Energy-difference block | $2n+6Mw-2w-2M+1$ | $18s_{\mathrm{SK}}+18w$ |
| Positive-part selector | $3w$ | $3$ |
| Cutoff-tail block | $2w+3$ | $14\ell_{j_{\mathrm{cut}}}+3w-3j_{\mathrm{cut}}-13$ |

The cutoff-tail row assumes the active-tail regime $3\leq j_{\mathrm{cut}}\leq w-1$.

The one-body signal Hamiltonian is

$$
H_{\mathrm{sig}}
=
\frac{1}{2}Z_{w-1}
-
\sum_{j=0}^{w-2}2^{j-w}Z_j,
$$

with

$$
H_{\mathrm{sig}}\ket{r}
=
(\eta-r)\ket{r},
\qquad
\eta
=
1+3\cdot2^{-w}.
$$

Table VII gives the phase-arithmetic resources:

| Component | Logical qubits | Non-Clifford depth |
|---|---:|---:|
| One-body $\mathrm{PREPARE}$ | $w$ | $2\ell_w$ |
| One-body $\mathrm{SELECT}$ | $2w$ | $0$ |
| Controlled one-body $\mathrm{SELECT}$ | $3w$ | $3$ |
| One-body reflection | $w+2$ | $14\ell_w-10$ |
| Controlled one-body reflection | $w+3$ | $14\ell_w-10$ |
| One-body qubitized operator | $2w+2$ | $18\ell_w-10$ |
| Controlled one-body qubitized operator | $3w+2$ | $18\ell_w-7$ |
| Square-root exponential arithmetic | $3w+2$ | $d_{\mathrm{hyb}}(54\ell_w-18)+3$ |

The degree is estimated numerically as

$$
d_{\mathrm{hyb}}
\approx
\left\lceil
\log\left(\frac{1}{\varepsilon_{\mathrm{tail}}}\right)
+
\log\left(\frac{1}{\varepsilon_{\mathrm{ops}}}\right)
+
1
\right\rceil.
$$

The complete hybrid coin has non-Clifford depth

$$
D_{\mathrm{hybrid}}
=
d_{\mathrm{hyb}}(54\ell_w-18)
+
36s_{\mathrm{SK}}
+
42w
-
6j_{\mathrm{cut}}
+
28\ell_{j_{\mathrm{cut}}}
-
17,
$$

and uses

$$
2n+6Mw-2M+2
$$

logical qubits.

This implementation is detailed in `3.2_hybrid_arithmetic.ipynb`.


<a id="proposal-preparation-unitaries"></a>
## Proposal preparation unitaries

### Uniform proposal

The uniform proposal acts on $\ket{x}_a\ket{0^n}_b$ by applying Hadamard gates to all qubits of register $b$:

$$
\ket{x}_a\ket{0^n}_b
\mapsto
\ket{x}_a
\frac{1}{\sqrt{2^n}}
\sum_y\ket{y}_b.
$$

It requires only the two system registers and has zero non-Clifford depth.

### Local proposal

The local proposal prepares a Dicke state of Hamming weight one on register $b$ and then XORs it with the current configuration using parallel CNOT gates:

$$
\ket{x}_a\ket{0^n}_b
\mapsto
\ket{x}_a
\frac{1}{\sqrt n}
\sum_{|z|=1}\ket{x\oplus z}_b.
$$

The CNOT layer is Clifford. For Hamming weight one, the Dicke-state construction used in the paper has non-Clifford depth bounded by

$$
2\left\lceil\log_2n\right\rceil.
$$

No qubits beyond the two $n$-qubit system registers are required.

### Hamiltonian-simulation proposal

The Hamiltonian-simulation proposal first loads the computational-basis label $x$ into register $b$ using parallel CNOT gates and then applies a Trotterized transverse-field Ising evolution. The Hamiltonian is

$$
H_{\mathrm{tf}}(\gamma)
=
H+\gamma H_X
=
H-\gamma\sum_iX_i,
$$

where $H$ is the diagonal SK Hamiltonian and $H_X=-\sum_iX_i$.

The symmetric second-order formula is

$$
U^{(r)}(t,\gamma)
=
\left(
e^{-itH/(2r)}
e^{-it\gamma H_X/r}
e^{-itH/(2r)}
\right)^r.
$$

At the level of the resource estimate, adjacent half-steps of the same commuting layer are merged. The dense $Z_iZ_j$ interaction terms are scheduled into at most $n$ parallel matching layers. Each Trotter step therefore contributes $n$ two-qubit interaction layers and three single-qubit rotation layers, giving non-Clifford depth

$$
r(n+3).
$$

The experiments use $r=50$. The randomized proposal is implemented by assigning a different pair $(t,\gamma)$ to each application of $W_\beta$, with

$$
\gamma\in(0.25,0.60),
\qquad
t\in(2.0,20.0).
$$

Table IV is therefore:

| Proposal block | Logical qubits | Non-Clifford depth |
|---|---:|---:|
| Uniform proposal | $2n$ | $0$ |
| Local proposal | $2n$ | $2\lceil\log_2n\rceil$ |
| Hamiltonian-simulation proposal | $2n$ | $r(n+3)$ |


<a id="reflection-and-accept-path-unitary"></a>
## Reflection and accept-path unitary

An $m$-controlled NOT is implemented with the two-clean-ancilla construction used in the paper. After decomposition into Clifford+$T$ gates, with each Toffoli assigned non-Clifford depth three, its depth is bounded by

$$
14\left\lceil\log_2m\right\rceil-10.
$$

For the hybrid Boltzmann coin, all temporary fixed-point and phase-arithmetic registers are returned to $\ket{0}$ within the coin unitary. The persistent coin register contains the GQSP control qubit and the $w$-qubit one-body selection register, so

$$
q_{\mathrm{coin}}
=
w+1.
$$

In this implementation, the accepted branch is the all-zero persistent coin state. The reflection is

$$
R_0
=
2\left(
I_A
\otimes
\ket{0^n}\!\bra{0^n}_B
\otimes
\ket{0^{w+1}}\!\bra{0^{w+1}}_{\mathrm{coin}}
\right)-I.
$$

The all-zero condition involves $n+w+1$ qubits, so the selective phase flip has $n+w$ controls. The reflection therefore has depth

$$
14\ell_{n+w}-10
$$

and uses $2n+w+3$ logical qubits.

The accept-path unitary is

$$
F\ket{x}_A\ket{y}_B\ket{z}_{\mathrm{coin}}
=
\begin{cases}
\ket{y}_A\ket{x}_B\ket{z}_{\mathrm{coin}},
&
z=0^{w+1},
\\
\ket{x}_A\ket{y}_B\ket{z}_{\mathrm{coin}},
&
z\neq0^{w+1}.
\end{cases}
$$

The circuit computes the zero-coin flag, fans it out using Clifford gates, applies the $n$ controlled swaps in parallel, and uncomputes the flag. Each controlled swap contains three sequential Toffoli gates, so the swap stage has non-Clifford depth nine. The total depth is

$$
28\ell_{w+1}-11.
$$

For $n\geq3$, the accept-path unitary uses $3n+w+1$ logical qubits.

Table VIII is:

| Component | Logical qubits | Non-Clifford depth |
|---|---:|---:|
| Reflection | $2n+w+3$ | $14\ell_{n+w}-10$ |
| Accept-path unitary | $3n+w+1$ | $28\ell_{w+1}-11$ |
